# 06 - Multi-Turn Staged Orchestrator (Intent -> Mechanics -> Narration)

This notebook is a dedicated **turn loop** example for DM orchestration.

It addresses the "single response then exit" issue by running a persistent session loop:
- player enters an action each turn
- model runs staged reasoning per turn
- tools can loop within each stage
- narration returns control to player for next turn

Stage model per turn:
1. **Intent stage**: clarify approach and stakes (minimal/no rolls)
2. **Mechanics stage**: optional checks only when uncertainty + stakes justify
3. **Narration stage**: resolve fiction and hand agency back to player

Key behavior goals:
- no railroading toward one obstacle
- no forced roll quotas
- dice are optional resolution tools, not default output


## Step 1 - Setup

Imports, adapter initialization, and runtime knobs.


In [ ]:
# Section: Setup
# Purpose: Import dependencies, configure trace controls, and initialize the adapter.

import json
import re
import sys
import time
from pathlib import Path
from typing import Any

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "orchestrator").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from orchestrator.llm_interaction.adapter import LLMAdapter, LLMError

MODEL = "gpt-oss:20b"
MAX_TURNS = 12
MAX_STAGE_ITERATIONS = 10
TRACE_PREVIEW_CHARS = 420
RAW_RESPONSE_PREVIEW_CHARS = 1400

# Readable-by-default trace settings.
# Set these to True/False as needed while debugging.
FULL_TRACE = False
SHOW_THINKING_TRACE = True
SHOW_RAW_RESPONSE = False
SHOW_RESPONSE_STATS = True
PRINT_FULL_TRACE_AFTER_RUN = False
PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN = False

adapter = LLMAdapter(model=MODEL, verbose=False)


## Step 2 - Character, World, and Runtime State

Defines source-of-truth context and mutable orchestration state.


In [ ]:
# Section: Runtime state
# Purpose: Define character/world data and mutable counters used across staged turns.

CHARACTER_SHEET = {
    "name": "Nyx Embervale",
    "class": "Rogue",
    "level": 5,
    "stats": {
        "strength": 10,
        "dexterity": 17,
        "constitution": 13,
        "intelligence": 14,
        "wisdom": 12,
        "charisma": 15,
    },
    "skills": {
        "acrobatics": "dexterity",
        "athletics": "strength",
        "deception": "charisma",
        "insight": "wisdom",
        "investigation": "intelligence",
        "perception": "wisdom",
        "persuasion": "charisma",
        "sleight_of_hand": "dexterity",
        "stealth": "dexterity",
        "survival": "wisdom",
    },
}

WORLD_STATE = {
    "location": "The Shattered Beacon",
    "scene_goal": "Recover the storm ledger from the upper archive.",
    "tension": "high",
    "time_of_day": "midnight",
    "hooks": [
        "A locked iron strongbox on a stone desk (optional lead)",
        "A tray with a suspicious wine carafe",
        "Faint scratch marks near a hidden wall panel",
        "Open storm maps pinned across a table",
        "Distant footsteps in the stairwell",
    ],
}

TODO_ACTIVE_STATUSES = {"pending", "in_progress"}
TODO_FINAL_STATUSES = {"done", "skipped", "blocked"}
TODO_ALLOWED_STATUSES = TODO_ACTIVE_STATUSES | TODO_FINAL_STATUSES

RUN_STATE = {
    "total_tool_calls": 0,
    "check_log": [],
    "session_notes": [],
    "last_observed": {},
    "rolls_requested": 0,
    "actions_prompted": 0,
    "turn_index": 0,
    "current_stage": "",
    "turn_roll_count": 0,
    "turn_todo": [],
    "turn_todo_summary": "",
    "turn_todo_revision": 0,
    "turn_todo_created_stage": "",
}

INITIAL_SCENE_PROMPT = (
    "Start in media res at the Shattered Beacon with improvisational pacing. "
    "Do not force outcomes, let player agency drive the narrative. Establish scene and ask what the player does."
)


def compute_turn_todo_counts(todo_items: list[dict[str, Any]] | None = None) -> dict[str, int]:
    items = RUN_STATE.get("turn_todo", []) if todo_items is None else todo_items
    counts = {
        "total": len(items),
        "pending": 0,
        "in_progress": 0,
        "done": 0,
        "skipped": 0,
        "blocked": 0,
    }
    for item in items:
        status = str(item.get("status", "pending")).strip().lower()
        if status in counts:
            counts[status] += 1
        else:
            counts["pending"] += 1
    return counts


def clear_turn_todo_for_new_turn() -> None:
    RUN_STATE["turn_todo"] = []
    RUN_STATE["turn_todo_summary"] = ""
    RUN_STATE["turn_todo_created_stage"] = ""


def reset_run_state() -> None:
    RUN_STATE["total_tool_calls"] = 0
    RUN_STATE["check_log"] = []
    RUN_STATE["session_notes"] = []
    RUN_STATE["last_observed"] = {}
    RUN_STATE["rolls_requested"] = 0
    RUN_STATE["actions_prompted"] = 0
    RUN_STATE["turn_index"] = 0
    RUN_STATE["current_stage"] = ""
    RUN_STATE["turn_roll_count"] = 0
    RUN_STATE["turn_todo_revision"] = 0
    clear_turn_todo_for_new_turn()


def ability_modifier(score: int) -> int:
    return (int(score) - 10) // 2


def world_summary() -> str:
    return json.dumps(WORLD_STATE, indent=2)


## Step 3 - Tool Implementations

Interactive tools:
- `prompt_player_action`: ask player for free-form intent
- `roll_dice`: DM-style roll call + player-entered d20 value


In [ ]:
# Section: Tool implementations
# Purpose: Provide state/intent/mechanics tools used by staged loop orchestration.

def read_character_sheet() -> dict[str, Any]:
    return json.loads(json.dumps(CHARACTER_SHEET))


def read_world_state() -> dict[str, Any]:
    return json.loads(json.dumps(WORLD_STATE))


def set_world_state(
    location: str | None = None,
    scene_goal: str | None = None,
    tension: str | None = None,
    time_of_day: str | None = None,
) -> dict[str, Any]:
    if location:
        WORLD_STATE["location"] = location
    if scene_goal:
        WORLD_STATE["scene_goal"] = scene_goal
    if tension:
        WORLD_STATE["tension"] = tension
    if time_of_day:
        WORLD_STATE["time_of_day"] = time_of_day
    return json.loads(json.dumps(WORLD_STATE))


def prompt_player_action(prompt: str = "What do you do?") -> dict[str, Any]:
    q = str(prompt).strip() or "What do you do?"

    RUN_STATE["actions_prompted"] += 1

    print("\n" + "-" * 72, flush=True)
    print("DM QUESTION", flush=True)
    print(q, flush=True)

    while True:
        action = input("Player action: ").strip()
        if action.lower() in {"q", "quit", "exit"}:
            raise RuntimeError("Player aborted during action prompt.")
        if action:
            break
        print("Please enter an action or 'q' to abort.", flush=True)

    RUN_STATE["last_observed"]["player_intent"] = action

    print(f"Captured action: {action}", flush=True)
    print("-" * 72 + "\n", flush=True)

    return {
        "player_action": action,
        "prompt": q,
    }


def _normalize_check_type(raw: Any) -> str:
    value = str(raw or "intent_check").strip().lower()
    allowed = {"forced_check", "scene_check", "intent_check"}
    if value not in allowed:
        return "intent_check"
    return value


def _normalize_todo_item(raw: Any, item_id: int) -> dict[str, Any]:
    if isinstance(raw, str):
        task = raw.strip()
        meta: dict[str, Any] = {}
    elif isinstance(raw, dict):
        task = str(raw.get("task") or raw.get("description") or "").strip()
        meta = raw
    else:
        raise ValueError(f"Invalid todo item at index {item_id}: expected object or string")

    if not task:
        raise ValueError(f"Todo item at index {item_id} must include non-empty 'task'")

    target_dc = meta.get("target_dc")
    if target_dc is not None:
        target_dc = int(target_dc)
        if target_dc < 5 or target_dc > 30:
            raise ValueError("target_dc must be between 5 and 30 when provided")

    item = {
        "id": int(item_id),
        "task": task,
        "requires_roll": bool(meta.get("requires_roll", False)),
        "suggested_skill": str(meta.get("suggested_skill", "")).strip().lower(),
        "check_type": _normalize_check_type(meta.get("check_type", "intent_check")),
        "target_dc": target_dc,
        "status": "pending",
        "resolution": "",
        "used_roll": False,
    }
    return item


def _find_turn_todo_item(item_id: int) -> dict[str, Any] | None:
    for item in RUN_STATE.get("turn_todo", []):
        if int(item.get("id", -1)) == int(item_id):
            return item
    return None


def set_turn_todo(items: list[dict[str, Any]] | list[str], plan_summary: str = "") -> dict[str, Any]:
    if not isinstance(items, list) or not items:
        raise ValueError("items must be a non-empty list")

    normalized: list[dict[str, Any]] = []
    for idx, raw in enumerate(items, start=1):
        normalized.append(_normalize_todo_item(raw, idx))

    RUN_STATE["turn_todo"] = normalized
    RUN_STATE["turn_todo_summary"] = str(plan_summary).strip()
    RUN_STATE["turn_todo_revision"] = int(RUN_STATE.get("turn_todo_revision", 0)) + 1
    RUN_STATE["turn_todo_created_stage"] = str(RUN_STATE.get("current_stage", ""))

    return {
        "ok": True,
        "revision": RUN_STATE["turn_todo_revision"],
        "summary": RUN_STATE["turn_todo_summary"],
        "items": json.loads(json.dumps(normalized)),
        "counts": compute_turn_todo_counts(normalized),
    }


def get_turn_todo(include_completed: bool = True) -> dict[str, Any]:
    items = RUN_STATE.get("turn_todo", [])
    if include_completed:
        filtered = items
    else:
        filtered = [x for x in items if str(x.get("status", "pending")) in TODO_ACTIVE_STATUSES]

    return {
        "ok": True,
        "revision": RUN_STATE.get("turn_todo_revision", 0),
        "summary": RUN_STATE.get("turn_todo_summary", ""),
        "created_stage": RUN_STATE.get("turn_todo_created_stage", ""),
        "items": json.loads(json.dumps(filtered)),
        "counts": compute_turn_todo_counts(items),
    }


def set_todo_item_status(
    item_id: int,
    status: str,
    resolution: str = "",
    used_roll: bool = False,
) -> dict[str, Any]:
    item = _find_turn_todo_item(int(item_id))
    if item is None:
        raise ValueError(f"Todo item {item_id} not found")

    normalized_status = str(status).strip().lower().replace(" ", "_")
    if normalized_status not in TODO_ALLOWED_STATUSES:
        raise ValueError(f"Invalid status '{status}'. Allowed: {sorted(TODO_ALLOWED_STATUSES)}")

    resolution_text = str(resolution).strip()

    if normalized_status == "done" and item.get("requires_roll") and not bool(used_roll):
        raise ValueError("requires_roll todo marked done must set used_roll=true")

    if normalized_status in {"done", "blocked"} and not resolution_text:
        raise ValueError(f"status '{normalized_status}' requires a non-empty resolution")

    item["status"] = normalized_status
    item["resolution"] = resolution_text
    item["used_roll"] = bool(used_roll)

    return {
        "ok": True,
        "item": json.loads(json.dumps(item)),
        "counts": compute_turn_todo_counts(),
    }


def get_skill_stat(skill: str) -> dict[str, Any]:
    key = str(skill).strip().lower()
    skills = CHARACTER_SHEET["skills"]
    if key not in skills:
        return {
            "ok": False,
            "error": f"Unknown skill '{skill}'. Available: {', '.join(sorted(skills.keys()))}",
        }
    return {"ok": True, "skill": key, "stat": skills[key]}


def get_stat_modifier(stat: str) -> dict[str, Any]:
    key = str(stat).strip().lower()
    stats = CHARACTER_SHEET["stats"]
    if key not in stats:
        return {
            "ok": False,
            "error": f"Unknown stat '{stat}'. Available: {', '.join(sorted(stats.keys()))}",
        }
    score = int(stats[key])
    return {
        "ok": True,
        "stat": key,
        "score": score,
        "modifier": ability_modifier(score),
    }


def roll_dice(
    skill: str,
    dc: int,
    modifier: int,
    check_type: str,
    dm_prompt: str = "",
    player_intent: str = "",
    consequence_on_fail: str = "",
) -> dict[str, Any]:
    skill_key = str(skill).strip().lower()
    target_dc = int(dc)
    mod = int(modifier)
    ctype = str(check_type).strip().lower()
    intent = str(player_intent).strip()
    fail_consequence = str(consequence_on_fail).strip()

    if dm_prompt.strip():
        prompt_text = dm_prompt.strip()
    elif ctype == "forced_check":
        prompt_text = f"This is a forced {skill_key} check. Roll now."
    elif ctype == "scene_check":
        prompt_text = f"Before you proceed, give me a quick {skill_key} check."
    else:
        prompt_text = f"Make a {skill_key} check." if not intent else f"Since you attempt '{intent}', make a {skill_key} check."

    RUN_STATE["rolls_requested"] += 1

    print("\n" + "=" * 72, flush=True)
    print("DM CALLS FOR A ROLL", flush=True)
    print(prompt_text, flush=True)
    print(f"Check type   : {ctype}", flush=True)
    print(f"Skill        : {skill_key}", flush=True)
    print(f"DC           : {target_dc}", flush=True)
    print(f"Modifier     : {mod:+d}", flush=True)
    if intent:
        print(f"Player intent: {intent}", flush=True)
    if fail_consequence:
        print(f"Fail state   : {fail_consequence}", flush=True)
    print("Enter raw d20 value (1-20), or 'q' to abort.", flush=True)

    while True:
        raw = input(f"d20 roll for {skill_key}: ").strip()
        if raw.lower() in {"q", "quit", "exit"}:
            raise RuntimeError("Player aborted manual roll input.")
        try:
            roll_value = int(raw)
        except ValueError:
            print("Invalid input. Enter an integer from 1 to 20.", flush=True)
            continue
        if not 1 <= roll_value <= 20:
            print("Out of range. Enter a value from 1 to 20.", flush=True)
            continue
        break

    total = roll_value + mod
    success = total >= target_dc

    result = {
        "check_type": ctype,
        "skill": skill_key,
        "dc": target_dc,
        "modifier": mod,
        "player_roll": roll_value,
        "total": total,
        "success": success,
        "player_intent": intent,
        "dm_prompt": prompt_text,
        "consequence_on_fail": fail_consequence,
        "turn_index": RUN_STATE.get("turn_index", 0),
        "stage": RUN_STATE.get("current_stage", ""),
    }

    RUN_STATE["last_observed"].update(
        {
            "check_type": ctype,
            "skill": skill_key,
            "dc": target_dc,
            "modifier": mod,
            "roll": roll_value,
            "total": total,
            "success": success,
            "player_intent": intent,
            "dm_prompt": prompt_text,
            "consequence_on_fail": fail_consequence,
        }
    )

    print(f"Recorded roll: d20={roll_value}, total={total}, success={success}", flush=True)
    print("=" * 72 + "\n", flush=True)

    return result


def record_skill_check(
    skill: str,
    stat: str,
    dc: int,
    roll: int,
    modifier: int,
    total: int,
    success: bool,
    check_type: str = "intent_check",
    player_intent: str = "",
    dm_prompt: str = "",
    consequence_on_fail: str = "",
    note: str = "",
) -> dict[str, Any]:
    entry = {
        "index": len(RUN_STATE["check_log"]) + 1,
        "turn_index": RUN_STATE.get("turn_index", 0),
        "check_type": str(check_type).strip().lower(),
        "skill": str(skill).strip().lower(),
        "stat": str(stat).strip().lower(),
        "dc": int(dc),
        "roll": int(roll),
        "modifier": int(modifier),
        "total": int(total),
        "success": bool(success),
        "player_intent": str(player_intent).strip(),
        "dm_prompt": str(dm_prompt).strip(),
        "consequence_on_fail": str(consequence_on_fail).strip(),
        "note": str(note).strip(),
    }
    RUN_STATE["check_log"].append(entry)
    if entry["note"]:
        RUN_STATE["session_notes"].append(entry["note"])
    return {"recorded": entry, "completed_checks": len(RUN_STATE["check_log"])}


def add_session_note(text: str) -> dict[str, Any]:
    note = str(text).strip()
    if not note:
        raise ValueError("text cannot be empty")
    RUN_STATE["session_notes"].append(note)
    return {"ok": True, "note": note, "total_notes": len(RUN_STATE["session_notes"])}


def get_progress() -> dict[str, Any]:
    checks = RUN_STATE["check_log"]
    latest = checks[-1] if checks else None
    return {
        "completed_checks": len(checks),
        "total_tool_calls": RUN_STATE["total_tool_calls"],
        "rolls_requested": RUN_STATE["rolls_requested"],
        "actions_prompted": RUN_STATE["actions_prompted"],
        "turn_index": RUN_STATE.get("turn_index", 0),
        "stage": RUN_STATE.get("current_stage", ""),
        "turn_roll_count": RUN_STATE.get("turn_roll_count", 0),
        "turn_todo_counts": compute_turn_todo_counts(),
        "turn_todo_revision": RUN_STATE.get("turn_todo_revision", 0),
        "latest_check": latest,
    }


## Step 4 - Tool Schemas and Runtime Policies

Policy goals:
- enforce stage boundaries (`intent`, `mechanics`, `narration`)
- prevent roll spam
- keep player agency open-ended


In [ ]:
# Section: Schemas + policies
# Purpose: Declare tool contracts and enforce staged, low-pressure mechanics behavior.

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "read_character_sheet",
            "description": "Read the full player character sheet.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "read_world_state",
            "description": "Read current world state.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_world_state",
            "description": "Update world state fields.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string"},
                    "scene_goal": {"type": "string"},
                    "tension": {"type": "string"},
                    "time_of_day": {"type": "string"},
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "prompt_player_action",
            "description": "Prompt player for an open-ended action declaration.",
            "parameters": {
                "type": "object",
                "properties": {"prompt": {"type": "string"}},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_turn_todo",
            "description": "Create or replace the turn todo list used by mechanics execution.",
            "parameters": {
                "type": "object",
                "properties": {
                    "plan_summary": {"type": "string"},
                    "items": {
                        "type": "array",
                        "minItems": 1,
                        "items": {
                            "type": "object",
                            "properties": {
                                "task": {"type": "string"},
                                "requires_roll": {"type": "boolean"},
                                "suggested_skill": {"type": "string"},
                                "check_type": {
                                    "type": "string",
                                    "enum": ["forced_check", "scene_check", "intent_check"],
                                },
                                "target_dc": {"type": "integer", "minimum": 5, "maximum": 30},
                            },
                            "required": ["task"],
                        },
                    },
                },
                "required": ["items"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_turn_todo",
            "description": "Read the current turn todo list and status counts.",
            "parameters": {
                "type": "object",
                "properties": {
                    "include_completed": {"type": "boolean"},
                },
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_todo_item_status",
            "description": "Update one todo item status after executing or deferring it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item_id": {"type": "integer", "minimum": 1},
                    "status": {
                        "type": "string",
                        "enum": ["pending", "in_progress", "done", "skipped", "blocked"],
                    },
                    "resolution": {"type": "string"},
                    "used_roll": {"type": "boolean"},
                },
                "required": ["item_id", "status"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_skill_stat",
            "description": "Resolve which stat applies to a skill.",
            "parameters": {
                "type": "object",
                "properties": {"skill": {"type": "string"}},
                "required": ["skill"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_stat_modifier",
            "description": "Resolve stat score and modifier.",
            "parameters": {
                "type": "object",
                "properties": {"stat": {"type": "string"}},
                "required": ["stat"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "roll_dice",
            "description": "Ask player for d20 input after DM calls for a check.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string"},
                    "dc": {"type": "integer", "minimum": 5, "maximum": 30},
                    "modifier": {"type": "integer", "minimum": -10, "maximum": 15},
                    "check_type": {
                        "type": "string",
                        "enum": ["forced_check", "scene_check", "intent_check"],
                    },
                    "dm_prompt": {"type": "string"},
                    "player_intent": {"type": "string"},
                    "consequence_on_fail": {"type": "string"},
                },
                "required": ["skill", "dc", "modifier", "check_type"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "record_skill_check",
            "description": "Record one resolved check.",
            "parameters": {
                "type": "object",
                "properties": {
                    "skill": {"type": "string"},
                    "stat": {"type": "string"},
                    "dc": {"type": "integer", "minimum": 5, "maximum": 30},
                    "roll": {"type": "integer", "minimum": 1, "maximum": 20},
                    "modifier": {"type": "integer", "minimum": -10, "maximum": 15},
                    "total": {"type": "integer", "minimum": -10, "maximum": 40},
                    "success": {"type": "boolean"},
                    "check_type": {"type": "string"},
                    "player_intent": {"type": "string"},
                    "dm_prompt": {"type": "string"},
                    "consequence_on_fail": {"type": "string"},
                    "note": {"type": "string"},
                },
                "required": ["skill", "stat", "dc", "roll", "modifier", "total", "success"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "add_session_note",
            "description": "Add a short persistent note.",
            "parameters": {
                "type": "object",
                "properties": {"text": {"type": "string"}},
                "required": ["text"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_progress",
            "description": "Read current orchestration progress and counters.",
            "parameters": {"type": "object", "properties": {}, "required": []},
        },
    },
]

TOOLS_BY_NAME = {tool["function"]["name"]: tool for tool in TOOLS}

STAGE_TOOL_ALLOWLIST = {
    "intent": [
        "read_character_sheet",
        "read_world_state",
        "set_world_state",
        "prompt_player_action",
        "set_turn_todo",
        "get_turn_todo",
        "add_session_note",
        "get_progress",
    ],
    "mechanics": [
        "read_character_sheet",
        "read_world_state",
        "set_world_state",
        "get_turn_todo",
        "set_todo_item_status",
        "get_skill_stat",
        "get_stat_modifier",
        "roll_dice",
        "record_skill_check",
        "add_session_note",
        "get_progress",
    ],
    "narration": [],
}


def tools_for_stage(stage_name: str) -> list[dict[str, Any]]:
    names = STAGE_TOOL_ALLOWLIST.get(str(stage_name).strip().lower(), [])
    return [TOOLS_BY_NAME[name] for name in names if name in TOOLS_BY_NAME]


def pre_tool_use(tool_name: str, args: dict[str, Any]) -> dict[str, Any]:
    stage = str(RUN_STATE.get("current_stage", "")).strip().lower()

    if stage in STAGE_TOOL_ALLOWLIST:
        allowed_tools = set(STAGE_TOOL_ALLOWLIST.get(stage, []))
        if tool_name not in allowed_tools:
            if stage == "narration":
                return {"allow": False, "reason": "Narration stage should not call tools."}
            return {"allow": False, "reason": f"{tool_name} is not available in {stage} stage."}

    if tool_name == "prompt_player_action" and stage != "intent":
        return {"allow": False, "reason": "prompt_player_action is only allowed in intent stage."}

    if tool_name == "set_turn_todo":
        if stage != "intent":
            return {"allow": False, "reason": "set_turn_todo is only allowed in intent stage."}

    if tool_name == "set_todo_item_status":
        if stage != "mechanics":
            return {"allow": False, "reason": "set_todo_item_status is only allowed in mechanics stage."}

        try:
            item_id = int(args.get("item_id", -1))
        except (TypeError, ValueError):
            return {"allow": False, "reason": "item_id must be an integer."}

        item = _find_turn_todo_item(item_id)
        if item is None:
            return {"allow": False, "reason": f"Todo item {item_id} does not exist."}

        status = str(args.get("status", "")).strip().lower().replace(" ", "_")
        if status not in TODO_ALLOWED_STATUSES:
            return {"allow": False, "reason": f"Invalid status '{status}' for todo item."}

        resolution = str(args.get("resolution", "")).strip()
        if status in {"done", "blocked"} and not resolution:
            return {"allow": False, "reason": f"status '{status}' requires a non-empty resolution."}

        if item.get("requires_roll") and status == "done" and not bool(args.get("used_roll", False)):
            return {"allow": False, "reason": "requires_roll todo marked done must set used_roll=true."}

    if tool_name == "roll_dice":
        if stage != "mechanics":
            return {"allow": False, "reason": "roll_dice is only allowed in mechanics stage."}

        dc = int(args.get("dc", 0))
        if dc < 5 or dc > 30:
            return {"allow": False, "reason": "DC must be between 5 and 30."}

        check_type = str(args.get("check_type", "")).strip().lower()
        allowed_types = {"forced_check", "scene_check", "intent_check"}
        if check_type not in allowed_types:
            return {"allow": False, "reason": f"check_type must be one of {sorted(allowed_types)}."}

        intent = str(args.get("player_intent", "")).strip()
        if check_type == "intent_check" and not intent:
            return {"allow": False, "reason": "intent_check requires player_intent after declared approach."}

        todo_items = RUN_STATE.get("turn_todo", [])
        if not todo_items:
            return {"allow": False, "reason": "No turn todo plan exists. Create it first with set_turn_todo."}

        pending_roll_items = [
            x for x in todo_items
            if str(x.get("status", "pending")) in TODO_ACTIVE_STATUSES and bool(x.get("requires_roll", False))
        ]
        if check_type != "forced_check" and not pending_roll_items:
            return {"allow": False, "reason": "No pending requires_roll todo item exists for this roll."}

        if RUN_STATE.get("turn_roll_count", 0) >= 2 and check_type != "forced_check":
            return {"allow": False, "reason": "Too many checks this turn. Resolve remaining items narratively if possible."}

    if tool_name == "record_skill_check" and "success" not in args:
        return {"allow": False, "reason": "record_skill_check requires explicit success boolean."}

    return {"allow": True}


def post_tool_use(tool_name: str, payload: dict[str, Any]) -> str | None:
    if payload.get("ok", True):
        return None

    if tool_name == "roll_dice":
        return "Roll input failed. Re-issue roll_dice with valid arguments and wait for player input."

    return "A tool failed. Correct arguments and continue."


def narration_stop_policy(assistant_text: str) -> str | None:
    text = (assistant_text or "").strip()
    lowered = text.lower()

    markers = [r"(?im)^\s*\d+[\.)]\s+", r"(?im)^\s*[-*]\s+"]
    for marker in markers:
        if re.search(marker, text):
            return "Do not provide menu-style options or bullet lists."

    banned_terms = ["choose one", "options:", "you can:", "pick one"]
    if any(term in lowered for term in banned_terms):
        return "Do not enumerate choices. Keep agency open-ended."

    open_prompt_phrases = ["what do you do", "how do you proceed", "what is your move", "what's your move", "your move"]
    has_open_prompt = any(p in lowered for p in open_prompt_phrases) or "?" in text
    if not has_open_prompt:
        return "End narration with an open-ended invitation for player action."

    return None


## Step 5 - Trace Helpers and Tool Dispatcher

Normalizes model responses, parses tool calls, executes tools, and tracks observed mechanics.


In [ ]:
# Section: Tracing + dispatch
# Purpose: Provide robust response parsing, tracing, and tool execution glue.

TOOL_IMPL = {
    "read_character_sheet": read_character_sheet,
    "read_world_state": read_world_state,
    "set_world_state": set_world_state,
    "prompt_player_action": prompt_player_action,
    "set_turn_todo": set_turn_todo,
    "get_turn_todo": get_turn_todo,
    "set_todo_item_status": set_todo_item_status,
    "get_skill_stat": get_skill_stat,
    "get_stat_modifier": get_stat_modifier,
    "roll_dice": roll_dice,
    "record_skill_check": record_skill_check,
    "add_session_note": add_session_note,
    "get_progress": get_progress,
}


def ts() -> str:
    return time.strftime("%H:%M:%S")


def compact(text: str) -> str:
    return " ".join(str(text).split())


def preview(text: str, limit: int = TRACE_PREVIEW_CHARS) -> str:
    if text is None:
        return ""
    s = str(text).strip()
    if FULL_TRACE or len(s) <= limit:
        return s
    return s[:limit] + f" ... [truncated {len(s) - limit} chars]"


def fmt_duration_ms(ns: Any) -> str:
    if isinstance(ns, (int, float)):
        return str(round(float(ns) / 1_000_000.0, 1))
    return "n/a"


def stage_tag(turn_index: int, stage_name: str) -> str:
    return f"T{turn_index}:{stage_name}"


def trace_print(tag: str, message: str, trace_lines: list[str] | None = None) -> None:
    line = f"[{ts()}] [{tag}] {message}"
    print(line, flush=True)
    if trace_lines is not None:
        trace_lines.append(line)


def response_to_dict(response: Any) -> dict[str, Any]:
    if hasattr(response, "model_dump"):
        try:
            data = response.model_dump(exclude_none=False)
            if isinstance(data, dict):
                return data
        except Exception:
            pass
    if isinstance(response, dict):
        return response
    return {"_type": type(response).__name__, "repr": repr(response)}


def extract_message_dict(response: Any) -> dict[str, Any]:
    data = response_to_dict(response)
    msg = data.get("message") if isinstance(data, dict) else None
    if isinstance(msg, dict):
        return msg
    if hasattr(msg, "model_dump"):
        try:
            dumped = msg.model_dump(exclude_none=False)
            if isinstance(dumped, dict):
                return dumped
        except Exception:
            pass
    return {}


def extract_thinking_text(response: Any) -> str:
    msg = extract_message_dict(response)
    thinking = msg.get("thinking", "")
    if isinstance(thinking, list):
        return "".join(str(x) for x in thinking)
    return str(thinking or "")


def normalize_tool_calls(response: Any) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    raw_calls = adapter._extract_tool_calls(response)

    for i, call in enumerate(raw_calls):
        if not isinstance(call, dict):
            continue

        fn = call.get("function", {})
        name = fn.get("name") or call.get("name")
        args = fn.get("arguments", {})

        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                args = {}

        if not isinstance(args, dict) or not name:
            continue

        out.append(
            {
                "id": call.get("id") or f"call_{i}",
                "name": str(name),
                "arguments": args,
                "raw": call,
                "source": "structured",
            }
        )

    return out


def extract_fallback_tool_calls_from_text(text: str) -> list[dict[str, Any]]:
    if not text:
        return []

    calls: list[dict[str, Any]] = []

    for line in text.splitlines():
        stripped = line.strip().rstrip(",")
        if not stripped.startswith("{") or not stripped.endswith("}"):
            continue
        try:
            obj = json.loads(stripped)
        except json.JSONDecodeError:
            continue
        if not isinstance(obj, dict):
            continue

        name = obj.get("name")
        params = obj.get("parameters", {})
        if not name or not isinstance(params, dict):
            continue

        calls.append(
            {
                "id": f"text_line_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {"function": {"name": str(name), "arguments": params}},
                "source": "text_fallback",
            }
        )

    pattern = re.compile(
        r'\{\s*"name"\s*:\s*"(?P<name>[^"]+)"\s*,\s*"parameters"\s*:\s*(?P<params>\{.*?\})\s*\}',
        flags=re.DOTALL,
    )

    for match in pattern.finditer(text):
        name = match.group("name")
        params_raw = match.group("params")
        try:
            params = json.loads(params_raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(params, dict):
            continue

        duplicate = any(c["name"] == str(name) and c["arguments"] == params for c in calls)
        if duplicate:
            continue

        calls.append(
            {
                "id": f"text_regex_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {"function": {"name": str(name), "arguments": params}},
                "source": "text_fallback",
            }
        )

    return calls


def update_last_observed(name: str, payload: dict[str, Any]) -> None:
    if not payload.get("ok"):
        return

    result = payload.get("result")
    if not isinstance(result, dict):
        return

    observed = RUN_STATE.setdefault("last_observed", {})

    if name == "prompt_player_action":
        action = result.get("player_action")
        observed["player_intent"] = action
        if isinstance(action, str):
            action_key = action.strip().lower()
            if action_key in CHARACTER_SHEET.get("skills", {}):
                observed["skill_hint"] = action_key

    elif name == "set_turn_todo":
        observed["turn_todo_revision"] = result.get("revision")
        observed["turn_todo_summary"] = result.get("summary", "")

    elif name == "set_todo_item_status":
        item = result.get("item")
        if isinstance(item, dict):
            observed["last_todo_item"] = item

    elif name == "get_skill_stat" and result.get("ok"):
        observed["skill"] = result.get("skill")
        observed["stat"] = result.get("stat")

    elif name == "get_stat_modifier" and result.get("ok"):
        observed["stat"] = result.get("stat")
        observed["modifier"] = result.get("modifier")

    elif name == "roll_dice":
        observed["check_type"] = result.get("check_type")
        observed["skill"] = result.get("skill")
        observed["dc"] = result.get("dc")
        observed["modifier"] = result.get("modifier")
        observed["roll"] = result.get("player_roll")
        observed["total"] = result.get("total")
        observed["success"] = result.get("success")
        observed["player_intent"] = result.get("player_intent")
        observed["dm_prompt"] = result.get("dm_prompt")
        observed["consequence_on_fail"] = result.get("consequence_on_fail")


def hydrate_record_skill_check_args(arguments: dict[str, Any]) -> dict[str, Any]:
    observed = RUN_STATE.get("last_observed", {})
    merged = dict(arguments)

    default_skill = observed.get("skill_hint", "perception")
    skill = str(merged.get("skill") or observed.get("skill") or default_skill).strip().lower()
    stat = str(merged.get("stat") or observed.get("stat") or CHARACTER_SHEET["skills"].get(skill, "wisdom")).strip().lower()
    dc = int(merged.get("dc", observed.get("dc", 12)))
    modifier = int(merged.get("modifier", observed.get("modifier", ability_modifier(CHARACTER_SHEET["stats"].get(stat, 10)))))
    roll = int(merged.get("roll", observed.get("roll", 10)))
    total = int(merged.get("total", observed.get("total", roll + modifier)))
    success = bool(merged.get("success", observed.get("success", total >= dc)))
    check_type = str(merged.get("check_type") or observed.get("check_type") or "intent_check").strip().lower()
    player_intent = str(merged.get("player_intent") or observed.get("player_intent") or "").strip()
    dm_prompt = str(merged.get("dm_prompt") or observed.get("dm_prompt") or "").strip()
    consequence_on_fail = str(merged.get("consequence_on_fail") or observed.get("consequence_on_fail") or "").strip()
    note = str(merged.get("note", "")).strip()

    return {
        "skill": skill,
        "stat": stat,
        "dc": dc,
        "roll": roll,
        "modifier": modifier,
        "total": total,
        "success": success,
        "check_type": check_type,
        "player_intent": player_intent,
        "dm_prompt": dm_prompt,
        "consequence_on_fail": consequence_on_fail,
        "note": note,
    }


def normalize_roll_dice_args(arguments: dict[str, Any], trace_lines: list[str]) -> dict[str, Any]:
    args = dict(arguments)
    stage = str(RUN_STATE.get("current_stage", "")).strip().lower()

    raw_type = str(args.get("check_type", "")).strip().lower()
    alias_map = {
        "skill": "intent_check",
        "skill_check": "intent_check",
        "check": "intent_check",
    }

    if raw_type in alias_map:
        args["check_type"] = alias_map[raw_type]
        trace_print("TOOL-NORMALIZE", f"roll_dice check_type '{raw_type}' -> '{args['check_type']}'", trace_lines)

    observed = RUN_STATE.get("last_observed", {})
    if not str(args.get("player_intent", "")).strip():
        observed_intent = str(observed.get("player_intent", "")).strip()
        if observed_intent:
            args["player_intent"] = observed_intent
            trace_print("TOOL-NORMALIZE", "roll_dice player_intent -> captured turn intent", trace_lines)

    if not str(args.get("check_type", "")).strip() and stage == "mechanics":
        pending_roll_items = [
            x
            for x in RUN_STATE.get("turn_todo", [])
            if str(x.get("status", "pending")) in TODO_ACTIVE_STATUSES and bool(x.get("requires_roll", False))
        ]

        if pending_roll_items:
            inferred = str(pending_roll_items[0].get("check_type", "")).strip().lower()
            if inferred in {"forced_check", "scene_check", "intent_check"}:
                args["check_type"] = inferred
                trace_print("TOOL-NORMALIZE", f"roll_dice check_type missing -> '{inferred}' (from todo item)", trace_lines)

        if not str(args.get("check_type", "")).strip() and str(args.get("player_intent", "")).strip():
            args["check_type"] = "intent_check"
            trace_print("TOOL-NORMALIZE", "roll_dice check_type missing -> 'intent_check' (from player intent)", trace_lines)

    return args


def normalize_todo_status_args(arguments: dict[str, Any], trace_lines: list[str]) -> dict[str, Any]:
    args = dict(arguments)

    raw_status = str(args.get("status", "")).strip().lower().replace(" ", "_")
    alias_map = {
        "complete": "done",
        "completed": "done",
        "finish": "done",
        "finished": "done",
        "in-progress": "in_progress",
        "inprogress": "in_progress",
        "skip": "skipped",
        "defer": "blocked",
        "deferred": "blocked",
    }
    normalized = alias_map.get(raw_status, raw_status)
    if normalized != raw_status:
        trace_print("TOOL-NORMALIZE", f"set_todo_item_status status '{raw_status}' -> '{normalized}'", trace_lines)
    args["status"] = normalized

    if normalized == "done" and "used_roll" not in args:
        item = None
        try:
            item = _find_turn_todo_item(int(args.get("item_id", -1)))
        except (TypeError, ValueError):
            item = None

        if item and item.get("requires_roll") and RUN_STATE.get("turn_roll_count", 0) > 0:
            args["used_roll"] = True
            trace_print("TOOL-NORMALIZE", "set_todo_item_status used_roll -> true (auto for requires_roll item)", trace_lines)

    return args


def execute_tool(name: str, arguments: dict[str, Any], trace_lines: list[str]) -> dict[str, Any]:
    RUN_STATE["total_tool_calls"] += 1
    call_no = RUN_STATE["total_tool_calls"]

    turn_idx = RUN_STATE.get("turn_index", 0)
    stage = RUN_STATE.get("current_stage", "")
    tag = stage_tag(turn_idx, stage)

    call_args = dict(arguments)

    if name == "roll_dice":
        call_args = normalize_roll_dice_args(call_args, trace_lines)

    if name == "set_todo_item_status":
        call_args = normalize_todo_status_args(call_args, trace_lines)

    if name == "record_skill_check":
        call_args = hydrate_record_skill_check_args(call_args)
        trace_print("TOOL-NORMALIZE", f"{tag} record_skill_check hydrated args={preview(json.dumps(call_args, ensure_ascii=True))}", trace_lines)

    trace_print("TOOL", f"{tag} #{call_no} CALL {name} args={preview(json.dumps(call_args, ensure_ascii=True))}", trace_lines)

    pre = pre_tool_use(name, call_args)
    trace_print("HOOK", f"{tag} pre {name} -> {json.dumps(pre, ensure_ascii=True)}", trace_lines)
    if not pre.get("allow", True):
        payload = {"ok": False, "error": pre.get("reason", "Blocked by pre_tool_use")}
        trace_print("TOOL", f"{tag} #{call_no} RESULT {name} -> {preview(json.dumps(payload, ensure_ascii=True))}", trace_lines)
        return payload

    fn = TOOL_IMPL.get(name)
    if fn is None:
        payload = {"ok": False, "error": f"Unknown tool: {name}"}
        trace_print("TOOL", f"{tag} #{call_no} RESULT {name} -> {payload['error']}", trace_lines)
        return payload

    try:
        payload = {"ok": True, "result": fn(**call_args)}
    except Exception as exc:
        payload = {"ok": False, "error": str(exc)}

    update_last_observed(name, payload)

    if name == "roll_dice" and payload.get("ok"):
        RUN_STATE["turn_roll_count"] = RUN_STATE.get("turn_roll_count", 0) + 1

    post_note = post_tool_use(name, payload)
    if post_note:
        payload["hook_note"] = post_note
        trace_print("HOOK", f"{tag} post {name} -> {post_note}", trace_lines)

    trace_print("TOOL", f"{tag} #{call_no} RESULT {name} -> {preview(json.dumps(payload, ensure_ascii=True))}", trace_lines)
    return payload


## Step 6 - Staged Orchestration Engine

Implements:
- base system prompt
- per-stage directives
- `run_stage(...)` with internal tool loop
- `run_turn(...)` using three stages
- `run_session(...)` multi-turn interactive loop


In [ ]:
# Section: Staged engine
# Purpose: Run a persistent multi-turn session with per-turn stage loops.

BASE_SYSTEM_PROMPT = (
    "You are an expert Dungeon Master for a role-playing game, focused on player agency, improvisation, and fair mechanics.\n\n"
    "Core rules:\n"
    "1. NEVER decide the player's choices, tactics, speech, or intent for them.\n"
    "2. NEVER provide options, bullet menus, or choose-one lists for player actions.\n"
    "3. Dice are optional outcome-resolution tools, only when uncertainty and stakes justify.\n"
    "4. Do not railroad toward one obstacle repeatedly; follow player intent naturally.\n"
    "5. Never fabricate roll outcomes, non-existent items, or NPC facts. Use tools for mechanics and world reference.\n"
)

STAGE_DIRECTIVES = {
    "intent": (
        "Stage: intent\n"
        "Goal: clarify player approach and define the execution todo list for this turn.\n"
        "- Prefer narrative framing over mechanics.\n"
        "- If intent is unclear, call prompt_player_action with an open question.\n"
        "- Build or refresh the plan with set_turn_todo using 1-5 concrete items.\n"
        "- Each todo item should represent one execution step mechanics can complete.\n"
        "- Mark requires_roll=true only when uncertainty + stakes justify a check.\n"
        "- Do not call roll_dice or record_skill_check in this stage.\n"
        "- End with: Intent Summary: <short summary>."
    ),
    "mechanics": (
        "Stage: mechanics\n"
        "Goal: execute the intent-stage todo list and resolve uncertainties.\n"
        "- Start by calling get_turn_todo(include_completed=false).\n"
        "- Iterate pending todo items and update each with set_todo_item_status.\n"
        "- For requires_roll items, use get_skill_stat -> get_stat_modifier -> roll_dice -> record_skill_check, then set_todo_item_status(..., status='done', used_roll=true).\n"
        "- For non-roll items, mark done/skipped/blocked with a concrete resolution.\n"
        "- Do not redefine the plan in mechanics; execute the plan created in intent.\n"
        "- End with exactly one line: Mechanics Todo Status: ALL_ITEMS_RESOLVED or Mechanics Todo Status: ITEMS_BLOCKED.\n"
        "- End with: Mechanics Summary: <what was resolved>."
    ),
    "narration": (
        "Stage: narration\n"
        "Goal: narrate consequences and hand control back to the player.\n"
        "- NEVER use menu-style options or bullets.\n"
        "- End with an invitation for player action."
    ),
}


def build_stage_system_prompt(stage_name: str) -> str:
    directive = STAGE_DIRECTIVES.get(stage_name, "")
    return BASE_SYSTEM_PROMPT + "\n" + directive


def serialize_turn_todo_for_handoff() -> str:
    items = RUN_STATE.get("turn_todo", [])
    slim = [
        {
            "id": x.get("id"),
            "task": x.get("task"),
            "requires_roll": x.get("requires_roll"),
            "status": x.get("status"),
            "suggested_skill": x.get("suggested_skill"),
            "check_type": x.get("check_type"),
            "target_dc": x.get("target_dc"),
        }
        for x in items
    ]
    return json.dumps(slim, ensure_ascii=True)


def build_stage_handoff_message(stage_name: str, player_input: str, stage_outputs: dict[str, str]) -> str:
    observed = RUN_STATE.get("last_observed", {})

    if stage_name == "mechanics":
        return (
            "Stage handoff: intent -> mechanics\n"
            f"Player input this turn: {player_input}\n"
            f"Captured intent: {observed.get('player_intent', '')}\n"
            f"Intent summary: {stage_outputs.get('intent', '')}\n"
            f"Turn todo summary: {RUN_STATE.get('turn_todo_summary', '')}\n"
            f"Turn todo items JSON: {serialize_turn_todo_for_handoff()}\n"
            f"Turn todo revision: {RUN_STATE.get('turn_todo_revision', 0)}\n"
            "Execute this todo list as-is in mechanics. Mark each item with set_todo_item_status before ending mechanics."
        )

    if stage_name == "narration":
        return (
            "Stage handoff: mechanics -> narration\n"
            f"Intent summary: {stage_outputs.get('intent', '')}\n"
            f"Mechanics summary: {stage_outputs.get('mechanics', '')}\n"
            f"Todo counts: {json.dumps(compute_turn_todo_counts(), ensure_ascii=True)}\n"
            "Narrate consequences of resolved/blocked todo items and end with open player agency."
        )

    return ""


def parse_mechanics_todo_status(text: str) -> str:
    lower = str(text or "").lower()
    match = re.search(r"mechanics todo status:\s*(all_items_resolved|items_blocked)\b", lower)
    if not match:
        return ""
    return match.group(1)


def run_stage(
    stage_name: str,
    messages: list[dict[str, Any]],
    turn_index: int,
    max_stage_iterations: int = MAX_STAGE_ITERATIONS,
    trace_lines: list[str] | None = None,
    response_snapshots: list[dict[str, Any]] | None = None,
) -> dict[str, Any]:
    if trace_lines is None:
        trace_lines = []
    if response_snapshots is None:
        response_snapshots = []

    RUN_STATE["current_stage"] = stage_name

    stage_start_rolls = RUN_STATE.get("rolls_requested", 0)
    stage_start_checks = len(RUN_STATE.get("check_log", []))
    stage_start_todo_revision = RUN_STATE.get("turn_todo_revision", 0)

    final_text = ""
    tag = stage_tag(turn_index, stage_name)
    stage_tools = tools_for_stage(stage_name)
    stage_tool_names = [x["function"]["name"] for x in stage_tools]

    trace_print("STAGE", f"{'=' * 14} {tag} START {'=' * 14}", trace_lines)
    trace_print("STAGE", f"{tag} tools={','.join(stage_tool_names) if stage_tool_names else '<none>'}", trace_lines)

    for stage_iter in range(1, max_stage_iterations + 1):
        trace_print("STAGE", f"{tag} iter {stage_iter}/{max_stage_iterations}", trace_lines)

        try:
            response = adapter.request_with_tools(
                stage=f"notebook_06_turn_{turn_index}_{stage_name}",
                system_prompt=build_stage_system_prompt(stage_name),
                messages=messages,
                tools=stage_tools,
            )
        except LLMError as exc:
            trace_print("MODEL-ERROR", f"{tag} {exc}", trace_lines)
            return {
                "status": "error",
                "error": str(exc),
                "stage": stage_name,
                "text": final_text,
                "messages": messages,
            }

        response_dict = response_to_dict(response)
        response_snapshots.append({"turn": turn_index, "stage": stage_name, "response": response_dict})

        if SHOW_RESPONSE_STATS:
            trace_print(
                "MODEL",
                (
                    f"{tag} done={response_dict.get('done_reason')} "
                    f"in_tok={response_dict.get('prompt_eval_count')} "
                    f"out_tok={response_dict.get('eval_count')} "
                    f"dur_ms={fmt_duration_ms(response_dict.get('total_duration'))}"
                ),
                trace_lines,
            )

        if SHOW_RAW_RESPONSE:
            raw = json.dumps(response_dict, ensure_ascii=True)
            trace_print("RAW", f"{tag} {preview(raw, RAW_RESPONSE_PREVIEW_CHARS)}", trace_lines)

        assistant_text = adapter._extract_content(response).strip()
        thinking_text = extract_thinking_text(response).strip()

        trace_print("ASSISTANT", f"{tag} content={preview(compact(assistant_text) or '<empty>')}", trace_lines)
        if SHOW_THINKING_TRACE:
            trace_print("THINKING", f"{tag} {preview(compact(thinking_text) or '<none>')}", trace_lines)

        tool_calls = normalize_tool_calls(response)
        if not tool_calls and stage_tools:
            fallback_source = "\n".join(x for x in [assistant_text, thinking_text] if x)
            fallback_calls = extract_fallback_tool_calls_from_text(fallback_source)
            if fallback_calls:
                tool_calls = fallback_calls
                trace_print("FALLBACK", f"{tag} recovered {len(tool_calls)} text-encoded tool calls", trace_lines)

        if tool_calls:
            messages.append(
                {
                    "role": "assistant",
                    "content": assistant_text,
                    "tool_calls": [c["raw"] for c in tool_calls],
                }
            )

            for call in tool_calls:
                trace_print("TOOL", f"{tag} source {call['name']} via {call.get('source', 'unknown')}", trace_lines)
                payload = execute_tool(call["name"], call["arguments"], trace_lines)
                messages.append(
                    {
                        "role": "tool",
                        "tool_name": call["name"],
                        "content": json.dumps(payload, separators=(",", ":"), ensure_ascii=True),
                    }
                )

                if payload.get("hook_note"):
                    messages.append({"role": "user", "content": "Runtime policy note: " + str(payload["hook_note"])})

            continue

        # No tool calls -> stage may finish, with stage-specific guardrails.
        if stage_name == "intent":
            text_lower = (assistant_text or "").lower()
            has_intent_summary = "intent summary:" in text_lower

            intent_block_reason = ""
            if not assistant_text.strip():
                intent_block_reason = "Intent stage returned empty output. Provide intent summary and turn todo plan."
            elif not has_intent_summary:
                intent_block_reason = "Intent stage must include 'Intent Summary: ...'."
            elif RUN_STATE.get("turn_todo_revision", 0) == stage_start_todo_revision:
                intent_block_reason = "Intent stage must call set_turn_todo to define execution items for this turn."
            elif not RUN_STATE.get("turn_todo", []):
                intent_block_reason = "Turn todo list is empty. Provide at least one actionable item."

            if intent_block_reason:
                trace_print("STAGE-BLOCK", f"{tag} {intent_block_reason}", trace_lines)
                messages.append({"role": "assistant", "content": assistant_text})
                messages.append(
                    {
                        "role": "user",
                        "content": "Intent stage blocked by runtime policy: " + intent_block_reason + " Continue and comply.",
                    }
                )
                continue

        if stage_name == "mechanics":
            rolls_delta = RUN_STATE.get("rolls_requested", 0) - stage_start_rolls
            checks_delta = len(RUN_STATE.get("check_log", [])) - stage_start_checks

            text_lower = (assistant_text or "").lower()
            has_mech_summary = "mechanics summary:" in text_lower
            todo_status = parse_mechanics_todo_status(assistant_text)

            counts = compute_turn_todo_counts()
            pending_count = counts.get("pending", 0) + counts.get("in_progress", 0)
            blocked_count = counts.get("blocked", 0)

            mechanics_block_reason = ""
            if not RUN_STATE.get("turn_todo", []):
                mechanics_block_reason = "Mechanics stage requires a turn todo list from intent stage."
            elif not assistant_text.strip():
                mechanics_block_reason = "Mechanics stage returned empty output. Provide todo status + summary."
            elif not todo_status:
                mechanics_block_reason = "Mechanics stage must declare 'Mechanics Todo Status: ALL_ITEMS_RESOLVED' or 'Mechanics Todo Status: ITEMS_BLOCKED'."
            elif not has_mech_summary:
                mechanics_block_reason = "Mechanics stage must include 'Mechanics Summary: ...'."
            elif pending_count > 0:
                mechanics_block_reason = (
                    f"Mechanics stage cannot finish with pending todo items ({pending_count} remaining). "
                    "Resolve or mark each item via set_todo_item_status."
                )
            elif blocked_count > 0 and todo_status != "items_blocked":
                mechanics_block_reason = "Todo contains blocked items, so status must be ITEMS_BLOCKED."
            elif blocked_count == 0 and todo_status != "all_items_resolved":
                mechanics_block_reason = "No blocked items remain, so status must be ALL_ITEMS_RESOLVED."
            elif todo_status == "items_blocked" and "because" not in text_lower:
                mechanics_block_reason = "ITEMS_BLOCKED requires a 'because' explanation in Mechanics Summary."
            elif todo_status == "all_items_resolved" and rolls_delta == 0 and checks_delta == 0 and any(x.get("requires_roll") for x in RUN_STATE.get("turn_todo", [])):
                mechanics_block_reason = "Todo included requires_roll items, but no roll/check was executed this stage."

            if mechanics_block_reason:
                trace_print("STAGE-BLOCK", f"{tag} {mechanics_block_reason}", trace_lines)
                messages.append({"role": "assistant", "content": assistant_text})
                messages.append(
                    {
                        "role": "user",
                        "content": "Mechanics stage blocked by runtime policy: " + mechanics_block_reason + " Continue and comply.",
                    }
                )
                continue

        if stage_name == "narration":
            policy_reason = narration_stop_policy(assistant_text)
            if policy_reason:
                trace_print("STAGE-BLOCK", f"{tag} {policy_reason}", trace_lines)
                messages.append({"role": "assistant", "content": assistant_text})
                messages.append(
                    {
                        "role": "user",
                        "content": "Narration style blocked by runtime policy: " + policy_reason + " Rewrite and continue.",
                    }
                )
                continue

        messages.append({"role": "assistant", "content": assistant_text})
        final_text = assistant_text
        trace_print("STAGE", f"{'=' * 14} {tag} END {'=' * 16}", trace_lines)
        return {
            "status": "completed",
            "stage": stage_name,
            "text": final_text,
            "messages": messages,
        }

    return {
        "status": "max_stage_iterations",
        "stage": stage_name,
        "text": final_text,
        "messages": messages,
    }


def reset_turn_observed_state() -> None:
    observed = RUN_STATE.setdefault("last_observed", {})
    for k in [
        "player_intent",
        "skill_hint",
        "skill",
        "stat",
        "dc",
        "modifier",
        "roll",
        "total",
        "success",
        "check_type",
        "dm_prompt",
        "consequence_on_fail",
        "turn_todo_revision",
        "turn_todo_summary",
        "last_todo_item",
    ]:
        observed.pop(k, None)


def run_turn(
    player_input: str,
    turn_index: int,
    messages: list[dict[str, Any]],
    trace_lines: list[str],
    response_snapshots: list[dict[str, Any]],
) -> dict[str, Any]:
    RUN_STATE["turn_index"] = turn_index
    RUN_STATE["turn_roll_count"] = 0
    clear_turn_todo_for_new_turn()
    reset_turn_observed_state()
    RUN_STATE.setdefault("last_observed", {})["player_intent"] = player_input

    messages.append({"role": "user", "content": f"Player action (turn {turn_index}): {player_input}"})

    stage_outputs: dict[str, str] = {}

    trace_print("TURN", f"{'#' * 18} TURN {turn_index} START {'#' * 18}", trace_lines)

    for stage_name in ["intent", "mechanics", "narration"]:
        if stage_name in {"mechanics", "narration"}:
            handoff = build_stage_handoff_message(stage_name, player_input, stage_outputs)
            if handoff:
                messages.append({"role": "user", "content": handoff})
                trace_print("HANDOFF", f"T{turn_index}:{stage_name} {preview(compact(handoff))}", trace_lines)

        out = run_stage(
            stage_name=stage_name,
            messages=messages,
            turn_index=turn_index,
            max_stage_iterations=MAX_STAGE_ITERATIONS,
            trace_lines=trace_lines,
            response_snapshots=response_snapshots,
        )

        if out.get("status") == "error":
            return {
                "status": "error",
                "turn_index": turn_index,
                "error": out.get("error", "stage error"),
                "messages": messages,
            }

        stage_text = out.get("text", "")
        stage_outputs[stage_name] = stage_text

        todo_counts = compute_turn_todo_counts()
        trace_print(
            "TURN",
            (
                f"T{turn_index}:{stage_name} done text={preview(compact(stage_text) or '<empty>')} "
                f"todo={json.dumps(todo_counts, ensure_ascii=True)}"
            ),
            trace_lines,
        )

    trace_print("TURN", f"{'#' * 18} TURN {turn_index} END {'#' * 20}", trace_lines)

    return {
        "status": "completed",
        "turn_index": turn_index,
        "stage_outputs": stage_outputs,
        "messages": messages,
    }


def run_session(
    initial_scene_prompt: str,
    max_turns: int = MAX_TURNS,
) -> dict[str, Any]:
    reset_run_state()

    trace_lines: list[str] = []
    response_snapshots: list[dict[str, Any]] = []
    start = time.time()

    messages: list[dict[str, Any]] = [
        {"role": "user", "content": "Character sheet (source of truth):\n" + json.dumps(CHARACTER_SHEET, indent=2)},
        {"role": "user", "content": "World state at scene start:\n" + world_summary()},
        {"role": "user", "content": "Scene start: " + initial_scene_prompt},
    ]

    trace_print("RUN", f"model={MODEL} max_turns={max_turns} max_stage_iterations={MAX_STAGE_ITERATIONS}", trace_lines)

    # Turn 0: establish opening narration.
    RUN_STATE["turn_index"] = 0
    intro = run_stage(
        stage_name="narration",
        messages=messages,
        turn_index=0,
        max_stage_iterations=MAX_STAGE_ITERATIONS,
        trace_lines=trace_lines,
        response_snapshots=response_snapshots,
    )

    if intro.get("status") == "error":
        return {
            "status": "error",
            "error": intro.get("error", "intro stage error"),
            "messages": messages,
            "trace_lines": trace_lines,
            "response_snapshots": response_snapshots,
            "progress": get_progress(),
            "check_log": list(RUN_STATE["check_log"]),
            "duration_s": round(time.time() - start, 3),
        }

    print("\n[DM OPENING]", intro.get("text", ""), flush=True)

    turn_results: list[dict[str, Any]] = []

    for turn_index in range(1, max_turns + 1):
        print("\n" + "#" * 72, flush=True)
        player_input = input(f"Player turn {turn_index} (type 'quit' to end): ").strip()
        if player_input.lower() in {"quit", "exit", "q"}:
            trace_print("RUN", f"player ended session at turn {turn_index}", trace_lines)
            break
        if not player_input:
            print("Please enter an action, or type 'quit' to end.", flush=True)
            continue

        turn_out = run_turn(
            player_input=player_input,
            turn_index=turn_index,
            messages=messages,
            trace_lines=trace_lines,
            response_snapshots=response_snapshots,
        )

        turn_results.append(turn_out)

        if turn_out.get("status") == "error":
            return {
                "status": "error",
                "error": turn_out.get("error", "turn error"),
                "messages": messages,
                "trace_lines": trace_lines,
                "response_snapshots": response_snapshots,
                "progress": get_progress(),
                "check_log": list(RUN_STATE["check_log"]),
                "turn_results": turn_results,
                "duration_s": round(time.time() - start, 3),
            }

        dm_text = turn_out.get("stage_outputs", {}).get("narration", "")
        print(f"\n[DM TURN {turn_index}] {dm_text}", flush=True)

    return {
        "status": "completed",
        "messages": messages,
        "trace_lines": trace_lines,
        "response_snapshots": response_snapshots,
        "progress": get_progress(),
        "check_log": list(RUN_STATE["check_log"]),
        "turn_results": turn_results,
        "duration_s": round(time.time() - start, 3),
    }


## Step 7 - Demo Run

Run this cell to start a multi-turn interactive session.

Controls:
- type actions in response to prompts
- type `quit` to stop the session loop


In [ ]:
# Section: Demo run
# Purpose: Start the staged multi-turn session loop and print run diagnostics.

result = run_session(initial_scene_prompt=INITIAL_SCENE_PROMPT, max_turns=MAX_TURNS)

print("\n===== SESSION RESULT =====")
print("status:", result.get("status"))
print("duration_s:", result.get("duration_s"))
print("progress:", json.dumps(result.get("progress", {}), indent=2))

print("\n===== CHECK LOG =====")
for entry in result.get("check_log", []):
    print(json.dumps(entry, ensure_ascii=True))

turn_results = result.get("turn_results", [])
print("\nTURN COUNT:", len(turn_results))
for out in turn_results:
    stage_outputs = out.get("stage_outputs", {})
    print(
        json.dumps(
            {
                "turn_index": out.get("turn_index"),
                "intent_summary": stage_outputs.get("intent", ""),
                "mechanics_summary": stage_outputs.get("mechanics", ""),
                "narration_preview": stage_outputs.get("narration", ""),
            },
            ensure_ascii=True,
        )
    )

trace_lines = result.get("trace_lines", [])
print("\nTRACE LINE COUNT:", len(trace_lines))
if PRINT_FULL_TRACE_AFTER_RUN:
    for line in trace_lines:
        print(line)

snapshots = result.get("response_snapshots", [])
print("\nRESPONSE SNAPSHOT COUNT:", len(snapshots))
if PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN:
    for i, snap in enumerate(snapshots, start=1):
        print(f"\n--- SNAPSHOT {i} ---")
        print(json.dumps(snap, indent=2, ensure_ascii=True))
